# 4 · AI Workflows & System Design for Finance (SEC/EDGAR)

**You leave with:** a Company Screening Engine that combines live SEC data, deterministic filtering, and validated LLM reasoning.

## The pattern of the day

```
INPUT → RETRIEVE → STRUCTURE → REASON → VALIDATE → HUMAN
         (code)     (code)     (model)    (code)    (you)
```

A **workflow** is a fixed plan written by you; the model fills designated steps. Three design rules carry everything:

1. **Code does math; the model does judgment.** Growth rates and filters live in pandas; the model writes grounded prose *about* them.
2. **Validate at the boundary.** Schema-forced output, plus a numeric audit: any figure in the prose that doesn't trace to your inputs gets flagged.
3. **The human gate is the exit.** Nothing is saved or sent without approval.

## SEC EDGAR in 60 seconds

Every US-listed company's filings, free, no key — just identify yourself (`SEC_EDGAR_USER_AGENT` in `.env`). **There:** 10-K/10-Q/8-K/20-F documents + XBRL fundamentals. **Not there:** prices, estimates. Gotchas we hit building this course (details: `session-04-workflows/edgar-cheatsheet.md`): companies drift between XBRL tags across years; foreign filers report IFRS in local currency; most pharma tags **no operating income** at all.

In [ ]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"repo root: {ROOT}")
print(f"API key:   {'configured' if HAS_KEY else 'NOT SET - cells that call Claude will be skipped'}")

## Part A — talk to EDGAR (follow along)

In [ ]:
from toolkit import edgar

fin = edgar.annual_financials("NVDA", n=3)
print(json.dumps(fin, indent=1)[:700], "...")

In [ ]:
# That's a real 10-K speaking. Let's compute metrics IN CODE (rule 1):
def metrics_row(ticker: str) -> dict | None:
    """3-year fundamentals -> one screening row. Returns None if unusable."""
    try:
        fin = edgar.annual_financials(ticker, n=3)
    except edgar.EdgarError as e:
        print(f"  {ticker}: skipped ({e})")
        return None
    rev = fin["revenue"]
    ni = {v["fy_end"]: v["val"] for v in fin["net_income"]}
    if len(rev) < 3 or fin["unit"] != "USD":
        return None
    r0, r1, r2 = (rev[i]["val"] for i in range(3))
    n_now, n_prev = ni.get(rev[2]["fy_end"]), ni.get(rev[1]["fy_end"])
    return {"ticker": fin["ticker"], "company": fin["company"], "fy_end": rev[2]["fy_end"],
            "revenue_bn": round(r2 / 1e9, 2),
            "growth_1y": round(r2 / r1 - 1, 4),
            "cagr_2y": round((r2 / r0) ** 0.5 - 1, 4),
            "net_margin": round(n_now / r2, 4) if n_now is not None else None,
            "net_margin_prior": round(n_prev / r1, 4) if n_prev is not None else None}

metrics_row("NVDA")

**Why NET margin and not operating margin?** Try `edgar.annual_values(edgar.get_company_facts("LLY"), edgar.OPERATING_INCOME_TAGS)` — most pharma present no operating subtotal, so the tag simply doesn't exist. Net income always does. *Data coverage is an analytical decision — make it in the open.*

## Part B — LAB: the Company Screening Engine

Universe: 16 industrials + pharma (`session-04-workflows/data/universe.csv`). First run fetches ~16 filings (a minute or two); everything caches.

### Exercise 1 — fetch the universe (defensive: one bad ticker must not kill the screen)

In [ ]:
import pandas as pd
universe = pd.read_csv(ROOT / "session-04-workflows" / "data" / "universe.csv")

### START CODE HERE ###
rows = [m for t in universe[None]                     # which column holds the tickers?
        if (m := metrics_row(None)) is not None]      # call the fetcher with what?
df = pd.DataFrame(None)                               # build the table from what?
### END CODE HERE ###

print(f"{len(df)} of {len(universe)} companies fetched")
df.round(3)

In [ ]:
# ✅ self-check — run me
assert len(df) >= 14, "expected at least 14 of 16 to fetch - is your loop skipping None rows?"
assert {"growth_1y", "net_margin"} <= set(df.columns)
assert df["net_margin"].abs().max() < 1, "margins should be decimals, not percents"
print("All checks passed ✅")

### Exercise 2 — the deterministic screen (pandas decides, not Claude)

Filter: `growth_1y >= min_growth` AND `net_margin >= min_margin`; if `require_improving`, also `net_margin > net_margin_prior`. Sort by growth, descending.

In [ ]:
def apply_screen(df, min_growth=0.08, min_margin=0.10, require_improving=False):
### START CODE HERE ###
    mask = (df[None] >= min_growth) & (df[None] >= min_margin)
    if require_improving:
        mask &= df["net_margin"] > df[None]               # improving vs which column?
    return df[mask].sort_values(None, ascending=False)    # rank the shortlist by what?
### END CODE HERE ###

shortlist = apply_screen(df)
print("Shortlist:", ", ".join(shortlist["ticker"]))
shortlist.round(3)

In [ ]:
# ✅ self-check — run me
assert len(shortlist) >= 1, "empty shortlist with default criteria - check your mask logic"
assert (shortlist["growth_1y"] >= 0.08).all() and (shortlist["net_margin"] >= 0.10).all()
tight = apply_screen(df, min_growth=0.10, require_improving=True)
assert len(tight) <= len(shortlist), "tighter criteria cannot grow the shortlist"
print("All checks passed ✅  Now play: change the criteria and watch the shortlist move.")

### Exercise 3 — grounded rationales + the numeric audit (needs API key)

ONE schema-forced call for the whole shortlist; the model may use **only** the metrics you send. Then audit its prose: any number that doesn't trace to your table gets flagged (`toolkit/verify.py`).

In [ ]:
from toolkit import llm, verify

RATIONALE_SCHEMA = {"type": "object", "required": ["rationales"], "properties": {
    "rationales": {"type": "array", "items": {"type": "object",
        "required": ["ticker", "observation"],
        "properties": {"ticker": {"type": "string"}, "observation": {"type": "string"}}}}}}

if HAS_KEY and len(shortlist):
### START CODE HERE ###
    result = llm.ask_structured(
        "Write a 2-3 sentence investment observation per company from these screening "
        "metrics (growth/margins as decimals). Use ONLY these metrics - no outside "
        f"knowledge, no new numbers:\n<metrics>\n{shortlist.to_json(orient='records')}\n</metrics>",
        name="record_rationales", schema=None,            # force which schema?
        system="You are screening companies for an investment committee. Grounded, direct, no hype.")
    source_vals = [v for r in shortlist.to_dict("records") for v in r.values()
                   if isinstance(v, (int, float))]
    for r in result["rationales"]:
        flags = verify.novel_numbers(None, None)          # audit WHICH text against WHICH numbers?
        mark = f"  ⚠️ untraceable: {flags}" if flags else "  ✅ grounded"
        print(f"\n{r['ticker']}: {r['observation']}{mark}")
### END CODE HERE ###
    print(f"\nToken usage: {llm.usage_summary()}")
else:
    print("No API key (or empty shortlist) - the deterministic screen above is still the deliverable core.")

## Part C — see the full workflow with the human gate

The complete 5-step version (with memo rendering, a sabotage-able validator, and the approval gate) lives in `session-04-workflows/demo/market_intel_workflow.py`. Run it from the terminal — dry-run needs no key:

```bash
python session-04-workflows/demo/market_intel_workflow.py NVDA --peers AMD INTC --dry-run
```

Worth trying: open `session-04-workflows/data/example_intel_memo.json`, change one figure, rerun — watch the validator catch your sabotage. Revert with `git checkout`.

## Deliverable checklist

- [ ] All ✅ checks green; you ran the screen with at least two different criteria sets
- [ ] You can answer: why did we screen on NET margin? and: which step would you never delegate to the model?
- [ ] (With key) rationales printed with the numeric-audit marks; anything ⚠️ judged by you
- [ ] Committed to your repo

**Next:** `05-agents.ipynb` — the model makes the plan. With a leash.